# Notebook 06 — Evaluación y Comparación de Modelos

**Responsable:** Estudiante C  
**Objetivo:** Comparar los tres modelos (Content-Based, MF+SGD, KNN) usando todas las métricas definidas en `src/evaluation/metrics.py`.

## Modelos a comparar
1. **Content-Based Filtering** (TF-IDF + Cosine Similarity)
2. **Matrix Factorization + SGD** (Collaborative Filtering, model-based)
3. **KNN Item-Item** (Collaborative Filtering, memory-based)

## Métricas
| Métrica | Descripción | Aplica a |
|---|---|---|
| RMSE | Error cuadrático medio en predicción de ratings | MF, KNN |
| MAE | Error absoluto medio | MF, KNN |
| Precision@K | Fracción relevantes en top-K | Todos |
| Recall@K | Fracción relevantes recuperados en top-K | Todos |
| F1@K | Media armónica P@K y R@K | Todos |
| MAP@K | Mean Average Precision | Todos |
| Coverage | % del catálogo recomendado | Todos |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, os.path.abspath('..'))

from src.models.matrix_factorization import MatrixFactorizationSGD
from src.models.knn import ItemKNN
from src.models.tfidf import TFIDFVectorizer
from src.models.cosine_similarity import cosine_similarity_matrix, get_top_n
from src.evaluation.metrics import rmse, mae, evaluate_model, precision_at_k, recall_at_k

## 1. Cargar Todos los Datos y Modelos

In [ ]:
# Matrices de ratings (CF)
train_matrix = np.load('../data/processed/ratings_train.npy')
test_matrix  = np.load('../data/processed/ratings_test.npy')
movies_cf    = pd.read_csv('../data/processed/movies_cf.csv')

# Dataset completo (Content-Based)
df_content   = pd.read_csv('../data/processed/movies_clean.csv')

n_users, n_items_cf = train_matrix.shape
print(f'Usuarios: {n_users} | Películas CF: {n_items_cf} | Películas CB: {len(df_content)}')

## 2. Entrenar (o Cargar) los Tres Modelos

In [ ]:
# --- Modelo 1: Content-Based ---
corpus = df_content['tag_soup'].fillna('').tolist()
vectorizer = TFIDFVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
sim_matrix_cb = cosine_similarity_matrix(tfidf_matrix)
print(f'Content-Based: {sim_matrix_cb.shape}')

# --- Modelo 2: MF + SGD ---
mf_model = MatrixFactorizationSGD(k=20, alpha=0.005, lambda_=0.02, epochs=100, seed=42)
mf_model.fit(train_matrix, verbose=False)
print(f'MF + SGD: entrenado')

# --- Modelo 3: KNN ---
knn_model = ItemKNN(k=20)
knn_model.fit(train_matrix)
print(f'KNN: entrenado')

## 3. Métricas de Predicción de Ratings (RMSE, MAE)

In [ ]:
t_users, t_movies = np.where(test_matrix > 0)
actual = test_matrix[t_users, t_movies]

# MF predicciones
mf_preds = np.sum(mf_model.P[t_users] * mf_model.Q[t_movies], axis=1)

# KNN predicciones
knn_preds_list = [knn_model.predict(u, i) for u, i in zip(t_users, t_movies)]
knn_preds = np.array(knn_preds_list)
knn_mask  = knn_preds > 0

print('=== Métricas de Predicción de Ratings ===')
print(f'{'Modelo':<20} {'RMSE':>8} {'MAE':>8}')
print('-' * 38)
print(f'{'MF + SGD':<20} {rmse(actual, mf_preds):>8.4f} {mae(actual, mf_preds):>8.4f}')
print(f'{'KNN (k=20)':<20} {rmse(actual[knn_mask], knn_preds[knn_mask]):>8.4f} {mae(actual[knn_mask], knn_preds[knn_mask]):>8.4f}')
print(f'{'Content-Based':<20} {"N/A":>8} {"N/A":>8}  (no predice ratings)')

## 4. Métricas de Ranking (Precision@K, Recall@K, F1@K)

In [ ]:
K = 10
# Umbral para definir "relevante": rating >= 7.0 en escala 1-10
RELEVANCE_THRESHOLD = 7.0

# Generar recomendaciones y relevancia para cada usuario
all_recs_mf  = []
all_recs_knn = []
all_relevant = []

for u in range(n_users):
    # Ítems relevantes en test (rating >= umbral)
    relevant = set(np.where((test_matrix[u] >= RELEVANCE_THRESHOLD))[0])
    all_relevant.append(relevant)

    # MF: top-K excluyendo lo ya visto en train
    mf_recs = mf_model.recommend(u, train_matrix, n=K)
    all_recs_mf.append(mf_recs)

    # KNN: top-K
    knn_recs = knn_model.recommend(u, n=K)
    all_recs_knn.append(knn_recs)

# Evaluación
metrics_mf  = evaluate_model(all_recs_mf,  all_relevant, n_items_cf, k=K)
metrics_knn = evaluate_model(all_recs_knn, all_relevant, n_items_cf, k=K)

print(f'\n=== Métricas de Ranking @K={K} (CF) ===')
print(f'{'Métrica':<15} {'MF+SGD':>10} {'KNN':>10}')
print('-' * 37)
for key in metrics_mf:
    print(f'{key:<15} {metrics_mf[key]:>10.4f} {metrics_knn[key]:>10.4f}')

## 5. Tabla Comparativa Final

In [ ]:
comparison = pd.DataFrame({
    'Modelo': ['Content-Based (TF-IDF)', 'MF + SGD', 'KNN Item-Item'],
    'Tipo': ['Content-Based', 'CF Model-Based', 'CF Memory-Based'],
    'RMSE': ['-', f'{rmse(actual, mf_preds):.4f}',
             f'{rmse(actual[knn_mask], knn_preds[knn_mask]):.4f}'],
    f'Precision@{K}': ['-', f'{metrics_mf[f"precision@{K}"]:.4f}',
                       f'{metrics_knn[f"precision@{K}"]:.4f}'],
    f'Recall@{K}':    ['-', f'{metrics_mf[f"recall@{K}"]:.4f}',
                       f'{metrics_knn[f"recall@{K}"]:.4f}'],
    f'F1@{K}':        ['-', f'{metrics_mf[f"f1@{K}"]:.4f}',
                       f'{metrics_knn[f"f1@{K}"]:.4f}'],
    'Coverage': [f'{(sim_matrix_cb > 0.1).any(axis=0).mean():.1%}',
                 f'{metrics_mf["coverage"]:.1%}',
                 f'{metrics_knn["coverage"]:.1%}'],
})

print('=== TABLA COMPARATIVA FINAL ===')
print(comparison.to_string(index=False))

## 6. Visualizaciones para el Informe

In [ ]:
# Gráfico: Precision@K y Recall@K para diferentes valores de K
k_range = [1, 5, 10, 15, 20]

prec_mf, rec_mf   = [], []
prec_knn, rec_knn = [], []

for k_val in k_range:
    # Regenerar recomendaciones para este k
    recs_mf_k  = [mf_model.recommend(u, train_matrix, n=k_val) for u in range(n_users)]
    recs_knn_k = [knn_model.recommend(u, n=k_val) for u in range(n_users)]

    prec_mf.append(np.mean([precision_at_k(r, rel, k_val) for r, rel in zip(recs_mf_k, all_relevant)]))
    rec_mf.append(np.mean([recall_at_k(r, rel, k_val) for r, rel in zip(recs_mf_k, all_relevant)]))
    prec_knn.append(np.mean([precision_at_k(r, rel, k_val) for r, rel in zip(recs_knn_k, all_relevant)]))
    rec_knn.append(np.mean([recall_at_k(r, rel, k_val) for r, rel in zip(recs_knn_k, all_relevant)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, prec_mf,  marker='o', label='MF + SGD', color='steelblue')
axes[0].plot(k_range, prec_knn, marker='s', label='KNN',      color='coral')
axes[0].set_title('Precision@K por modelo')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Precision@K')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, rec_mf,  marker='o', label='MF + SGD', color='steelblue')
axes[1].plot(k_range, rec_knn, marker='s', label='KNN',      color='coral')
axes[1].set_title('Recall@K por modelo')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Recall@K')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparación MF+SGD vs KNN — Métricas de Ranking', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/precision_recall_comparison.png', dpi=150)
plt.show()

## 7. Discusión

Completar con análisis basado en los resultados obtenidos:

**Content-Based Filtering:**
- Ventaja: no necesita ratings históricos (no hay cold-start para películas nuevas)
- Limitación: recomendaciones muy similares al ítem consultado ("filter bubble")
- Coverage: alto, ya que puede recomendar cualquier película del catálogo

**Matrix Factorization + SGD:**
- Ventaja: captura patrones latentes en el comportamiento de usuarios
- Ventaja: el RMSE más bajo indica mejor predicción de ratings
- Limitación: cold-start (usuarios/películas sin ratings no pueden ser modelados)
- El gráfico de convergencia en notebook 04 muestra que SGD converge efectivamente

**KNN Item-Item:**
- Ventaja: altamente interpretable ("porque le gustó X, le recomendamos Y")
- Limitación: O(n²) en memoria y precalcómputo — no escala bien a catálogos grandes
- Coverage: puede ser bajo si la matriz de ratings es muy sparse

*(Reemplazar con análisis cuantitativo de los resultados reales)*